In [0]:
#-----------------------------------------
# DAR59 Awaab's Law
#-----------------------------------------
# Cohort Summary:
# Population: 0–15 year olds
# ICD-10 Filter: Codes starting with "J"
#-----------------------------------------
# RDE Tables:
# 'rde_all_problems'       - used for cohort selection based on recorded problems
# 'rde_all_diagnosis'      - used for cohort selection based on diagnosis codes
# 'rde_encounter'          - contains clinical encounter details and service information
# 'rde_patient_demographics'- contains patient demographic information (e.g., age, sex)
#
# Mapping Tables:
# 'map_address'            - provides derived information such as LOSA, IMD quintile, and partial postcode
# 'map_patient_journey'    - contains patient-level clinical journey and service interactions
#
# OMOP Table:
# 'death'                  - contains death information for the patient
#
# Air Quality Table:
# 'air_quality'            - contains the derived environmental exposure data relevant to the study
#-----------------------------------------


# Project configuration
project_identifier = 'dar059'

rde_tables = [
    'rde_all_problems',
    'rde_all_diagnosis',
    'rde_encounter',
    'rde_patient_demographics'
]

map_tables = ['map_patient_journey']

omop_tables = ['death']

max_ig_risk = 3
max_ig_severity = 2
columns_to_exclude = ['ADC_UPDT','LATITUDE','LONGITUDE',"UPRN"]

# Create cohort view
cohort_sql = f"""
CREATE OR REPLACE VIEW 6_mgmt.cohorts.{project_identifier} AS
WITH resp_cohort AS (
    SELECT DISTINCT Person_ID
    FROM 4_prod.rde.rde_all_diagnosis
    WHERE Diagnosis_code ILIKE 'J%'

    UNION

    SELECT DISTINCT Person_ID
    FROM 4_prod.rde.rde_all_problems
    WHERE Problem_code ILIKE 'J%'
)
SELECT DISTINCT
    pd.Person_ID AS PERSON_ID,
    CAST(NULL AS STRING) AS SUBCOHORT
FROM 4_prod.rde.rde_patient_demographics pd
JOIN resp_cohort rc
    ON pd.Person_ID = rc.Person_ID
WHERE FLOOR(MONTHS_BETWEEN(CURRENT_DATE(), pd.Date_of_Birth) / 12) BETWEEN 0 AND 15;
"""
spark.sql(cohort_sql)


spark.sql("USE CATALOG 5_projects")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS 5_projects.{project_identifier}")

# Get list of existing views in the target schema
existing_views_df = spark.sql(f"""
    SHOW VIEWS IN 5_projects.{project_identifier}
""")

# Drop all existing views in the schema
if existing_views_df.count() > 0:
    for row in existing_views_df.collect():
        view_name = row.viewName
        spark.sql(f"DROP VIEW IF EXISTS {project_identifier}.{view_name}")
        print(f"Dropped view: {project_identifier}.{view_name}")


def get_columns_with_high_tags(catalog_name, schema_name, table_name):
    query = f"""
        SELECT column_name
        FROM {catalog_name}.information_schema.column_tags
        WHERE schema_name = '{schema_name}'
          AND table_name = '{table_name}'
          AND (
                (tag_name = 'ig_risk' AND tag_value > {max_ig_risk})
             OR (tag_name = 'ig_severity' AND tag_value > {max_ig_severity})
          )
    """

    df = spark.sql(query).toPandas()
    return df['column_name'].unique().tolist()

# Function to get column names excluding specified columns and columns with high tags

def get_columns_except_excluded(catalog_name, schema_name, table_name):
    full_table_path = f"{catalog_name}.{schema_name}.{table_name}"

    # All columns
    all_columns = spark.table(full_table_path).columns

    # Columns with high-risk tags
    tagged_columns = get_columns_with_high_tags(catalog_name, schema_name, table_name)

    # Combine dynamic + static exclusions
    excluded = set(tagged_columns) | set(columns_to_exclude)

    # Filter
    filtered = [c for c in all_columns if c not in excluded]

    return ", ".join(filtered)


# Function to determine the person ID column name

def find_person_id_column(catalog_name, schema_name, table_name):
    full_table_path = f"{catalog_name}.{schema_name}.{table_name}"
    columns = spark.table(full_table_path).columns

    # Common possible names
    potential_columns = [
        'PERSON_ID', 'person_id', 'Person_ID', 'personid',
        'PERSONID', 'PersonID', 'participant_id'
    ]

    # Exact match
    for col in potential_columns:
        if col in columns:
            return col

    # Fuzzy
    for col in columns:
        c = col.lower()
        if "person" in c and "id" in c:
            return col

    return None


#----------- RDE Table Processing -----------

catalog_name = "4_prod"
schema_name = "rde"

for table_name in rde_tables:

    # Build full path once
    full_table_path = f"{catalog_name}.{schema_name}.{table_name}"

    columns = get_columns_except_excluded(catalog_name, schema_name, table_name)
    person_id_col = find_person_id_column(catalog_name, schema_name, table_name)

    if person_id_col:
        view_sql = f"""
        CREATE OR REPLACE VIEW 5_projects.{project_identifier}.{table_name} AS
        WITH source_data AS (
            SELECT {columns}
            FROM {full_table_path}
        )
        SELECT s.*
        FROM source_data s
        INNER JOIN 6_mgmt.cohorts.{project_identifier} c
            ON s.{person_id_col} = c.PERSON_ID
        """
    else:
        print(f"Warning: No person ID column found in rde.{table_name}. Creating view without cohort filtering.")

        view_sql = f"""
        CREATE OR REPLACE VIEW 5_projects.{project_identifier}.{table_name} AS
        SELECT {columns}
        FROM {full_table_path}
        """

    spark.sql(view_sql)
    print(f"Created view: 5_projects.{project_identifier}.{table_name}")


#----------- Omop & Bronze Table Processing -----------

def process_and_create_views(tables, source_catalog, source_schema, project_identifier):
    """
    Generic function to create cohort-filtered views for a list of tables.
    It uses helper functions:
      - find_person_id_column(catalog, schema, table)
      - get_columns_except_excluded(catalog, schema, table)
    """

    for table_name in tables:
        
        # Build full table path
        full_table_path = f"{source_catalog}.{source_schema}.{table_name}"
        
        # Identify person ID column
        person_id_col = find_person_id_column(source_catalog, source_schema, table_name)
        
        # Get filtered column list (comma-separated string)
        column_list = get_columns_except_excluded(source_catalog, source_schema, table_name)

        # Prefix with table alias 'm.'
        prefixed_columns = ", ".join([f"m.{c.strip()}" for c in column_list.split(",")])
        
        if person_id_col:
            view_sql = f"""
            CREATE OR REPLACE VIEW 5_projects.{project_identifier}.{table_name} AS
            SELECT {prefixed_columns}
            FROM {full_table_path} m
            INNER JOIN 6_mgmt.cohorts.{project_identifier} c
                ON m.{person_id_col} = c.PERSON_ID
            """

            spark.sql(view_sql)
            print(f"Created cohort-filtered view: 5_projects.{project_identifier}.{table_name}")

        else:
            print(f"⚠ Warning: No person ID column in {full_table_path}. Creating unfiltered view.")

            view_sql = f"""
            CREATE OR REPLACE VIEW 5_projects.{project_identifier}.{table_name} AS
            SELECT {column_list}
            FROM {full_table_path}
            """

            spark.sql(view_sql)
            print(f"Created unfiltered view: 5_projects.{project_identifier}.{table_name}")

process_and_create_views(map_tables, '4_prod', 'bronze', project_identifier)
process_and_create_views(omop_tables, '4_prod', 'omop', project_identifier)

#----------- Map_address Table Processing -----------

columns = get_columns_except_excluded("4_prod", "bronze", "map_address")

spark.sql(f"""
    CREATE OR REPLACE VIEW 5_projects.{project_identifier}.map_address AS
    SELECT {columns}
    FROM 4_prod.bronze.map_address m
    INNER JOIN 6_mgmt.cohorts.{project_identifier} c
        ON m.PARENT_ENTITY_ID = c.PERSON_ID
    WHERE m.PARENT_ENTITY_NAME = 'PERSON'
""")

#----------- Air quality Table Processing -----------

from typing import Optional, Any
from pyspark.sql import SparkSession, Window, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.functions import col, row_number


# Constants
CATALOG_AQ_MONTHLY = "3_lookup.geo.uk_air_quality_monthly"
CATALOG_AQ_SITES = "3_lookup.geo.uk_air_quality_sites"
CATALOG_ADDRESS = "4_prod.bronze.map_address"

MAX_RADIUS_KM = 25.0  # 3-site interpolation radius
NEAR_RADIUS_KM = 2.0  # single-site fallback radius
EPS_KM = 0.1  # distance floor to avoid infinite weights (100 m)

# List of pollutant columns expected in monthly table
POLLUTANT_COLS = [
    "no", "no2", "nox", "o3", "so2", "co",
    "pm10", "pm2_5", "v10", "v2_5", "nv10", "nv2_5"
]

def _haversine_km_col(lat1, lon1, lat2, lon2):
    """
    Great-circle distance in kilometers as a pure column expression.
    lat/lon must be in degrees (float columns).
    """
    # Earth radius in km (WGS84 mean radius)
    R = F.lit(6371.0088)

    # Convert to radians
    phi1 = F.radians(lat1)
    phi2 = F.radians(lat2)
    dphi = F.radians(lat2 - lat1)
    dlambda = F.radians(lon2 - lon1)

    a = (
        F.sin(dphi / 2) * F.sin(dphi / 2)
        + F.cos(phi1) * F.cos(phi2) * F.sin(dlambda / 2) * F.sin(dlambda / 2)
    )
    c = 2 * F.atan2(F.sqrt(a), F.sqrt(F.lit(1) - a))
    return R * c


def estimate_pollutant_for_person(
    spark: SparkSession
) -> DataFrame:
    """
    Compute pollutant estimates for cohort persons' addresses.

    Logic (per address × pollutant × day):
      1) If there are ≥ 3 sites within 25 km: take the 3 closest and compute inverse-square weighted estimate.
      2) Else, if ≥ 1 site within 2 km: take the nearest site's measured value.
      3) Else: no estimate (NULL).

    Returns:
      DataFrame with columns:
        address_id, pollutant, aq_obs_dt, estimate_value, method
      where method ∈ {"tri_weighted_25km", "nearest_within_2km"}.
    """

    # Cohort (distinct persons)
    cohort = spark.sql("SELECT DISTINCT PERSON_ID FROM 6_mgmt.cohorts.dar059").alias("c")

    # Addresses for those persons
    addr = (
        spark.table(CATALOG_ADDRESS).alias("a")
        .join(cohort,
              (col("a.PARENT_ENTITY_NAME") == F.lit("PERSON")) & (col("a.PARENT_ENTITY_ID") == col("c.PERSON_ID")),
              how="inner")
        .filter(col("a.LATITUDE").isNotNull() & col("a.LONGITUDE").isNotNull())
        .select(
        col("a.ADDRESS_ID").alias("ADDRESS_ID"),
        col("a.LATITUDE").alias("LATITUDE"),
        col("a.LONGITUDE").alias("LONGITUDE"))
        .distinct()
    )

    # Monthly AQ (to long format per site × date × pollutant)
    aq_month = spark.table(CATALOG_AQ_MONTHLY).select("site_id", "date", *POLLUTANT_COLS)

    # Use Spark's stack() to unpivot pollutant columns
    expr = "stack({}, {}) AS (pollutant, value)".format(
        len(POLLUTANT_COLS),
        ", ".join(f"'{c}', {c}" for c in POLLUTANT_COLS)
    )

    aq_long = (
        aq_month
        .select("site_id", "date", F.expr(expr))
        .filter(col("value").isNotNull())
        .withColumnRenamed("date", "aq_obs_dt")
    )

    # Cross-join addresses × sites and compute distance (km)
    sites = (
        spark.table(CATALOG_AQ_SITES)
        .select(
            "site_id",
            col("latitude").alias("site_lat"),
            col("longitude").alias("site_lon"),
            "source", "code", "site", "site_type"
        )
        .filter(col("site_lat").isNotNull() & col("site_lon").isNotNull())
    )

    addr_sites = (
        sites.crossJoin(addr)
        .withColumn(
            "distance_km",
            _haversine_km_col(
                col("site_lat"), col("site_lon"),
                col("LATITUDE"), col("LONGITUDE")
            )
        )
    )

    # Join AQ and filter within 25 km
    within_25 = addr_sites.filter(col("distance_km") <= F.lit(MAX_RADIUS_KM))

    aq_within_25 = (
        within_25.alias("s")
        .join(aq_long.alias("a"), col("s.site_id") == col("a.site_id"), "inner")
        .select(
            col("s.ADDRESS_ID").alias("address_id"),
            col("s.site_id").alias("site_id"),
            col("s.distance_km").alias("distance_km"),
            col("a.pollutant").alias("pollutant"),
            col("a.value").alias("value"),
            col("a.aq_obs_dt").alias("aq_obs_dt")
        )
        .dropDuplicates()
    )

    # Require ≥ 3 sites per address × pollutant × day
    w_cnt = Window.partitionBy("address_id", "pollutant", "aq_obs_dt")
    aq_25_ge3 = (
        aq_within_25
        .withColumn("site_count", F.count(F.lit(1)).over(w_cnt))
        .filter(col("site_count") >= F.lit(3))
    )

    # Pick 3 closest sites (row_number over distance)
    w_nearest3 = (
        Window
        .partitionBy("address_id", "pollutant", "aq_obs_dt")
        .orderBy(col("distance_km").asc())
    )

    closest3 = (
        aq_25_ge3
        .withColumn("rn", row_number().over(w_nearest3))
        .filter(col("rn") <= F.lit(3))
    )

    # Inverse-square weighting (w = 1 / max(d, EPS)^2)
    weighted3 = (
        closest3
        .withColumn("safe_d", F.greatest(col("distance_km"), F.lit(EPS_KM)))
        .withColumn("w", F.lit(1.0) / (col("safe_d") * col("safe_d")))
        .withColumn("wv", col("value") * col("w"))
    )

    estimate_25km = (
        weighted3
        .groupBy("address_id", "pollutant", "aq_obs_dt")
        .agg((F.sum("wv") / F.sum("w")).alias("estimate_value"))
        .withColumn("method", F.lit("tri_weighted_25km"))
    )

    # Fallback: nearest site within 2 km when < 3 sites in 25 km
    within_2 = addr_sites.filter(col("distance_km") <= F.lit(NEAR_RADIUS_KM))

    aq_within_2 = (
        within_2.alias("s")
        .join(aq_long.alias("a"), col("s.site_id") == col("a.site_id"), "inner")
        .select(
            col("s.ADDRESS_ID").alias("address_id"),
            col("s.site_id").alias("site_id"),
            col("s.distance_km").alias("distance_km"),
            col("a.pollutant").alias("pollutant"),
            col("a.value").alias("value"),
            col("a.aq_obs_dt").alias("aq_obs_dt")
        )
        .dropDuplicates()
    )

    w_nearest = (
        Window
        .partitionBy("address_id", "pollutant", "aq_obs_dt")
        .orderBy(col("distance_km").asc())
    )

    nearest_2km = (
        aq_within_2
        .withColumn("rn", row_number().over(w_nearest))
        .filter(col("rn") == F.lit(1))
        .select(
            "address_id", "pollutant", "aq_obs_dt",
            col("value").alias("estimate_value")
        )
        .withColumn("method", F.lit("nearest_within_2km"))
    )

    #   Do a left-anti of nearest_2km where tri estimate exists, then union so each address × pollutant × day has at most one row.
    key_cols = ["address_id", "pollutant", "aq_obs_dt"]

    # Nearest2 rows that do NOT have a tri estimate for the same key
    nearest_only = (
        nearest_2km.alias("n")
        .join(estimate_25km.alias("t"), on=key_cols, how="left_anti")
    )

    final_estimate = estimate_25km.unionByName(nearest_only)

    return final_estimate

final_df = estimate_pollutant_for_person(spark)

final_df.write.mode("overwrite").saveAsTable(
    f"5_projects.{project_identifier}.air_quality"
)

# Create schema view
schema_sql = f"""
CREATE OR REPLACE VIEW 5_projects.{project_identifier}.schema AS
SELECT 
    table_name,
    column_name,
    COALESCE(comment, '') as column_commentqq
FROM 5_projects.information_schema.columns
WHERE table_catalog = '5_projects'
AND table_schema = '{project_identifier}'
AND table_name != 'schema'
ORDER BY table_name, column_name
"""
spark.sql(schema_sql)
print(f"Created schema view: 5_projects.{project_identifier}.schema")